# CrowdSight Training Notebook

Works on **Google Colab** and **Kaggle Notebooks** (T4 / P100 free tier).

**Quick start:**
1. Runtime → Change runtime type → **GPU (T4)**
2. Run all cells (Runtime → Run all)
3. Checkpoints saved to `/content/drive/MyDrive/crowdsight/` (Colab) or `/kaggle/working/` (Kaggle)

| Model | Est. training time (T4) | MAE target |
|---|---|---|
| CSRNet + DM-Count | ~3 hours (200 epochs) | ~59.7 |
| CLIP-EBC | ~1.5 hours (200 epochs) | ~56 |
| MAC-CNN | ~2 hours (200 epochs) | ~80 |

## 0 · Detect environment

In [ ]:
import os, sys

IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle')

print('Environment:', 'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Other')

import subprocess
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                           '--format=csv,noheader'], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print('GPU:', gpu_info.stdout.strip())
else:
    print('⚠️  No GPU detected — switch runtime to GPU before training!')

## 1 · Mount Google Drive (Colab only)
Skip this cell on Kaggle.

In [ ]:
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/crowdsight'
else:
    SAVE_DIR = '/kaggle/working/crowdsight'

os.makedirs(SAVE_DIR, exist_ok=True)
print('Checkpoints will be saved to:', SAVE_DIR)

## 2 · Install dependencies

In [ ]:
%%bash
pip install -q \
    transformers>=4.35.0 \
    huggingface_hub>=0.20.0 \
    scipy>=1.11.0 \
    h5py>=3.9.0 \
    tqdm>=4.66.0 \
    mlflow>=2.10.0
echo "Done"

## 3 · Clone CrowdSight

In [ ]:
REPO_URL = 'https://github.com/rasuljon-dev/crowdsight.git'  # ← update if needed
WORK_DIR = '/content/crowdsight' if IS_COLAB else '/kaggle/working/crowdsight'

if not os.path.exists(WORK_DIR):
    os.system(f'git clone {REPO_URL} {WORK_DIR}')
else:
    os.system(f'git -C {WORK_DIR} pull')

os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
print('Working directory:', os.getcwd())

## 4 · Download ShanghaiTech dataset

**Kaggle users:** Add the dataset `tthien/shanghaitech` in the right panel instead of running this cell.

**Colab users:** Run the cell below — downloads from Google Drive public share (~150 MB).

In [ ]:
DATA_ROOT = os.path.join(WORK_DIR, 'data', 'ShanghaiTech')

# ── Kaggle: dataset is already mounted ────────────────────────────────────
if IS_KAGGLE:
    # When using `tthien/shanghaitech` Kaggle dataset, files land here:
    KAGGLE_DATA = '/kaggle/input/shanghaitech'
    if os.path.exists(KAGGLE_DATA):
        os.makedirs(os.path.dirname(DATA_ROOT), exist_ok=True)
        os.system(f'ln -sf {KAGGLE_DATA} {DATA_ROOT}')
        print('Linked Kaggle dataset →', DATA_ROOT)
    else:
        print('Add the tthien/shanghaitech dataset in the Kaggle sidebar first!')

# ── Colab: download via gdown ──────────────────────────────────────────────
elif IS_COLAB:
    os.makedirs(DATA_ROOT, exist_ok=True)
    os.system('pip install -q gdown')
    import gdown
    # ShanghaiTech Part A — public mirror
    SHA_ID = '16dhJn72OzaGRR8kNQLuDyXRTiep_DTTG'
    SHA_ZIP = '/content/sha.zip'
    if not os.path.exists(SHA_ZIP):
        gdown.download(id=SHA_ID, output=SHA_ZIP, quiet=False)
    os.system(f'unzip -q {SHA_ZIP} -d {DATA_ROOT}')
    print('ShanghaiTech extracted to', DATA_ROOT)

print('\nDataset structure:')
os.system(f'ls {DATA_ROOT}')

## 5 · Configure training

In [ ]:
# ── Choose your model ─────────────────────────────────────────────────────
#   'csrnet'   → best CNN accuracy (MAE ~59.7)  — recommended first run
#   'clip_ebc' → VLM backbone, only head trained (MAE ~56)  — fastest
#   'mac_cnn'  → original thesis baseline (MAE ~80)
MODEL   = 'csrnet'       # ← change this

#   'mse'     → fast, standard
#   'dmcount' → OT + TV regularisation, better MAE (recommended for csrnet)
LOSS    = 'dmcount'      # ← change this

EPOCHS     = 200
BATCH_SIZE = 8
LR         = 1e-5 if MODEL == 'csrnet' else 1e-4   # higher LR ok for CLIP-EBC
DATASET    = 'shanghaitech_a'

# CLIP-EBC specific
BLOCK_SIZE = 64
NUM_BINS   = 21

CKPT_DIR = os.path.join(SAVE_DIR, f'{MODEL}_{LOSS}')
os.makedirs(CKPT_DIR, exist_ok=True)

print(f'Model      : {MODEL}')
print(f'Loss       : {LOSS}')
print(f'Epochs     : {EPOCHS}')
print(f'Batch size : {BATCH_SIZE}')
print(f'LR         : {LR}')
print(f'Checkpoints: {CKPT_DIR}')

## 6 · Train

In [ ]:
cmd = (
    f'python -m train.train '
    f'--model {MODEL} '
    f'--loss {LOSS} '
    f'--dataset {DATASET} '
    f'--data_root {DATA_ROOT} '
    f'--epochs {EPOCHS} '
    f'--batch_size {BATCH_SIZE} '
    f'--lr {LR} '
    f'--checkpoint_dir {CKPT_DIR} '
    f'--block_size {BLOCK_SIZE} '
    f'--num_bins {NUM_BINS} '
    f'--num_workers 2 '
    f'--save_every 20'
)
print('Running:', cmd)
os.system(cmd)

## 7 · Benchmark

In [ ]:
BEST_CKPT   = os.path.join(CKPT_DIR, 'best.pth')
RESULTS_DIR = os.path.join(SAVE_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

bench_cmd = (
    f'python benchmark.py '
    f'--checkpoint {BEST_CKPT} '
    f'--dataset {DATASET} '
    f'--data_root {DATA_ROOT} '
    f'--output {RESULTS_DIR}/{MODEL}_{LOSS}.json'
)
print('Running:', bench_cmd)
os.system(bench_cmd)

## 8 · Export lean weights for release

In [ ]:
RELEASE_DIR = os.path.join(SAVE_DIR, 'release')
os.makedirs(RELEASE_DIR, exist_ok=True)

release_cmd = (
    f'python release/export_weights.py '
    f'--input {BEST_CKPT} '
    f'--output {RELEASE_DIR}/{MODEL}_{LOSS}.pth '
    f'--metrics {RESULTS_DIR}/{MODEL}_{LOSS}.json'
)
print('Running:', release_cmd)
os.system(release_cmd)
print('\nRelease weight saved to:', f'{RELEASE_DIR}/{MODEL}_{LOSS}.pth')

## 9 · (Optional) Push to HuggingFace Hub

In [ ]:
# Fill in your HuggingFace token and repo ID to push weights automatically.
# Get your token at: https://huggingface.co/settings/tokens

HF_TOKEN   = ''                               # ← paste your HF token
HF_REPO_ID = 'YOUR_USERNAME/crowdsight'       # ← your HF repo

if HF_TOKEN and HF_REPO_ID:
    hub_cmd = (
        f'python release/export_weights.py '
        f'--input {BEST_CKPT} '
        f'--output {RELEASE_DIR}/{MODEL}_{LOSS}.pth '
        f'--hub {HF_REPO_ID} '
        f'--hf_token {HF_TOKEN}'
    )
    os.system(hub_cmd)
else:
    print('Skipping HF Hub push — fill in HF_TOKEN and HF_REPO_ID above.')

## 10 · Quick inference test

In [ ]:
import sys
sys.path.insert(0, WORK_DIR)

from api.inference import CrowdInferenceEngine
from PIL import Image
import numpy as np
import base64, io
import matplotlib.pyplot as plt

engine = CrowdInferenceEngine(weights_path=BEST_CKPT, model_name=MODEL)
print(f'Device: {engine.device_name}  |  Weights loaded: {engine.weights_loaded}')

# Test on a random image (replace with a real crowd image for meaningful results)
arr = (np.random.rand(480, 640, 3) * 255).astype('uint8')
img = Image.fromarray(arr)

result = engine.analyze(img)
print(f"Count: {result['count']}  |  Inference: {result['inference_time_ms']:.1f} ms")

heatmap = Image.open(io.BytesIO(base64.b64decode(result['density_map'])))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.imshow(img);      ax1.set_title('Input');   ax1.axis('off')
ax2.imshow(heatmap);  ax2.set_title(f'Density map  (count={result["count"]:.0f})'); ax2.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'sample_output.png'), dpi=120)
plt.show()